# Stage 2: Supervised Fine-Tuning (SFT)
This notebook executes the Supervised Fine-Tuning phase, loading either the adapter checkpoint from Stage 1 or starting directly from the base model, mapping SFT templates to teach the model how to follow instruction-based user support prompts.

To load a previously fine-tuned model (either a full model or an adapter), you'll typically use the `transformers` library, potentially with `peft` if you used LoRA or similar adapter-based tuning.

First, make sure you have the necessary libraries installed:

In [1]:
# Install libraries in Google Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes
!pip install unsloth_zoo
!pip install datasets
!pip install trl

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sflrdnom/unsloth_647ad3123a23472bab9f21e65ba5bdc5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sflrdnom/unsloth_647ad3123a23472bab9f21e65ba5bdc5
  Resolved https://github.com/unslothai/unsloth.git to commit d105bd7b42ea8d4ecdb3e3364abb605b558c017e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.8 MB/s eta 0:00:00
  

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from unsloth import FastLanguageModel


max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load the base model first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


# Prepare the model for further PEFT operations (e.g., if resuming training)
# This step is crucial to make the LoRA layers trainable for SFT.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
prompt_format = """Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}"""

EOS_TOKEN = tokenizer.eos_token
def format_prompts(examples):
    instructions = examples["instruction"]
    responses    = examples["response"]
    texts = []
    for inst, resp in zip(instructions, responses):
        text = prompt_format.format(instruction=inst, response=resp) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [6]:
from datasets import load_dataset

# Load the instruction dataset (upload 'instruction_dataset.jsonl' via the sidebar)
dataset = load_dataset("json", data_files="/content/drive/MyDrive/AIML-2026/instruction_dataset.jsonl", split="train")
dataset = dataset.map(format_prompts, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

In [7]:

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, # Changed from 2 to 1 to avoid PicklingError with multiprocessing
    packing = False,
     args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # Disable saving to bypass pickling issues during checkpoint creation
        report_to = "none", # Disable reporting to avoid potential issues with logger pickling
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/105 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 105 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.830175
10,2.690232
15,1.675916
20,1.387461
25,1.340099
30,1.186589
35,1.150027
40,1.050484
45,1.016290
50,0.922126


In [8]:
# Inference test
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    prompt_format.format(
        instruction = "My payment went through, but I am still not upgraded. Why?",
        response = "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 150, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Below is an instruction that describes a customer support scenario for practiceyourspeech.com. Write a response that appropriately completes the request.

### Instruction:
My payment went through, but I am still not upgraded. Why?

### Response:
Check your speech count in the Subscriptions tab. If it's free, you need to upgrade to Pro.


In [9]:
# Save SFT model
model.save_pretrained("sft_lora_model")
tokenizer.save_pretrained("sft_lora_model")

Unsloth: Restored added_tokens_decoder metadata in sft_lora_model/tokenizer_config.json.


('sft_lora_model/tokenizer_config.json', 'sft_lora_model/tokenizer.json')

In [10]:
# Load the SFT adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "sft_lora_model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False,
)
# Save as the merged base for DPO
model.save_pretrained_merged("/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged", tokenizer, save_method="merged_16bit")

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Detected local model directory: /content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage1-merged


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [03:12<00:00, 192.77s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [14:05<00:00, 845.78s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged`


In [11]:
import os

# Define the path to your merged model directory
merged_model_path = "/content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged"

# List the contents of the directory
if os.path.exists(merged_model_path):
    print(f"Contents of {merged_model_path}:")
    for item in os.listdir(merged_model_path):
        print(item)
else:
    print(f"Directory not found: {merged_model_path}")

Contents of /content/drive/MyDrive/AIML-2026/qwen2.5-7b-stage2-merged:
tokenizer_config.json
tokenizer.json
config.json
generation_config.json
model.safetensors
